In [ ]:
Sources: Wikipedia (Population), World Bank API (GDP per capita), Worldometers (Population), Global Data Lab (Temperature)

In [ ]:
## 1. Wikipedia – Population

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = 'https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population'

headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(URL, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', {'class': 'wikitable'})

rows = []
for tr in table.find_all('tr')[1:]:
    cols = tr.find_all(['td'])
    if len(cols) < 2:
        continue
    country = cols[0].get_text(strip=True)
    population_raw = cols[1].get_text(strip=True)
    population_clean = population_raw.replace(',', '').split('[')[0].strip()
    try:
        population = int(population_clean)
    except ValueError:
        population = None
    rows.append({'country': country, 'population': population})

df_wikipedia = pd.DataFrame(rows)
df_wikipedia = df_wikipedia[df_wikipedia['country'] != 'World']
print(f'Total rows: {len(df_wikipedia)}')
print(f'Missing values: {df_wikipedia["population"].isna().sum()}')
df_wikipedia.head()

In [ ]:
df_wikipedia.to_csv('../../data/raw/population_wikipedia.csv', index=False)
print('Saved to data/raw/population_wikipedia.csv')

In [ ]:
## 2. World Bank API – GDP per Capita

In [ ]:
BASE_URL = 'https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD'
YEAR = 2023

params = {'format': 'json', 'per_page': 300, 'date': YEAR}
response = requests.get(BASE_URL, params=params)
response.raise_for_status()
data = response.json()

records = data[1]

rows = []
for entry in records:
    country = entry.get('country', {}).get('value')
    country_code = entry.get('countryiso3code')
    gdp_value = entry.get('value')
    rows.append({'country': country, 'country_code': country_code, 'gdp_per_capita_usd': gdp_value})

# Filter out aggregates (regions, world totals etc.)
country_resp = requests.get('http://api.worldbank.org/v2/country', params={'format': 'json', 'per_page': 300})
country_data = country_resp.json()[1]
valid_codes = {c['id'] for c in country_data if c['region']['value'] != 'Aggregates'}

df_gdp = pd.DataFrame(rows)
df_gdp = df_gdp[df_gdp['country_code'].isin(valid_codes)]

na_count = df_gdp['gdp_per_capita_usd'].isna().sum()
print(f'Total rows: {len(df_gdp)}')
print(f'Missing GDP values: {na_count}')
if na_count > 0:
    print(df_gdp[df_gdp['gdp_per_capita_usd'].isna()][['country', 'country_code']].to_string())
df_gdp.head()

In [ ]:
df_gdp.to_csv('../../data/raw/gdp_per_capita.csv', index=False)
print('Saved to data/raw/gdp_per_capita.csv')

In [ ]:
## 3. Worldometers – Population 2026

In [ ]:
URL_WM = 'https://www.worldometers.info/world-population/population-by-country/'

response = requests.get(URL_WM)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')
table = soup.find('table')

rows = []
for tr in table.find('tbody').find_all('tr'):
    cols = tr.find_all('td')
    if len(cols) < 3:
        continue
    country = cols[1].text.strip()
    population = cols[2].text.strip().replace(',', '')
    try:
        population = int(population)
    except:
        population = None
    rows.append({'country': country, 'population': population})

df_worldometers = pd.DataFrame(rows)
na_count = df_worldometers['population'].isna().sum()
print(f'Total rows: {len(df_worldometers)}')
print(f'Missing values: {na_count}')
df_worldometers.head()

In [ ]:
df_worldometers.to_csv('../../data/raw/worldometers_population.csv', index=False)
print('Saved to data/raw/worldometers_population.csv')

In [ ]:
## 4. Global Data Lab – Annual Temperature

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

URL_TEMP = 'https://globaldatalab.org/geos/table/surfacetempyear/?levels=1&interpolation=0&extrapolation=0'

options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get(URL_TEMP)
time.sleep(5)

rows = []
table_rows = driver.find_elements(By.CSS_SELECTOR, 'table tbody tr')

for row in table_rows:
    cols = row.find_elements(By.TAG_NAME, 'td')
    if len(cols) < 2:
        continue
    country = cols[0].text.split('(')[0].strip()
    temps = [c.text for c in cols[1:]]
    years = list(range(1990, 1990 + len(temps)))
    for year, val in zip(years, temps):
        try:
            val = float(val)
        except:
            val = None
        rows.append({'country': country, 'year': year, 'temperature': val})

driver.quit()

df_temp = pd.DataFrame(rows)
na_count = df_temp['temperature'].isna().sum()
print(f'Total rows: {len(df_temp)}')
print(f'Missing temperature values: {na_count}')
df_temp.head()

In [ ]:
df_temp.to_csv('../../data/raw/temperature.csv', index=False)
print('Saved to data/raw/temperature.csv')